In [2]:
from pathlib import Path
import json
import pandas as pd

current_folder = Path.cwd()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

with (project_root / "models" / "modeling_config.json").open(
    encoding="utf-8"
) as file:
    modeling_config = json.load(file)

modeling_table = pd.read_parquet(
    project_root / "data" / "processed"
    / "household_campaign_modeling.parquet"
)

feature_columns = modeling_config["numeric_features"]
candidate_feature_columns = modeling_config["campaign_type_features"]

print(f"Modeling examples: {len(modeling_table):,}")
print(f"Candidate inputs: {len(candidate_feature_columns)}")
print(candidate_feature_columns)

Modeling examples: 7,091
Candidate inputs: 7
['baskets_90d', 'retailer_sales_90d', 'recency_days_90d', 'has_transactions_90d', 'avg_retailer_sales_per_basket_90d', 'active_days_90d', 'DESCRIPTION']


In [3]:
validation_start_day = modeling_config["validation_start_day"]
test_start_day = modeling_config["test_start_day"]

train_data = modeling_table.loc[
    modeling_table["END_DAY"] < validation_start_day
].copy()

validation_data = modeling_table.loc[
    (modeling_table["START_DAY"] >= validation_start_day)
    & (modeling_table["END_DAY"] < test_start_day)
].copy()

test_data = modeling_table.loc[
    modeling_table["START_DAY"] >= test_start_day
].copy()

print(f"Training examples: {len(train_data):,}")
print(f"Validation examples: {len(validation_data):,}")
print(f"Test examples: {len(test_data):,}")

Training examples: 2,499
Validation examples: 1,584
Test examples: 2,218


In [4]:
X_train = train_data[candidate_feature_columns].copy()
y_train = train_data[modeling_config["target"]].copy()

X_validation = validation_data[candidate_feature_columns].copy()
y_validation = validation_data[modeling_config["target"]].copy()

print(f"Training input shape: {X_train.shape}")
print(f"Validation input shape: {X_validation.shape}")

Training input shape: (2499, 7)
Validation input shape: (1584, 7)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

xgb_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        feature_columns
    ),
    (
        "campaign_type",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ["DESCRIPTION"]
    )
])

xgb_pipeline = Pipeline([
    ("preprocessor", xgb_preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42,
        n_jobs=4
    ))
])

print("XGBoost pipeline defined; not trained yet.")

XGBoost pipeline defined; not trained yet.


In [6]:
xgb_pipeline.fit(X_train, y_train)

print("XGBoost training complete.")
print(
    "Boosting rounds:",
    xgb_pipeline.named_steps["model"].get_booster().num_boosted_rounds()
)

XGBoost training complete.
Boosting rounds: 200


In [7]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

xgb_validation_probabilities = xgb_pipeline.predict_proba(
    X_validation
)[:, 1]

xgb_validation_metrics = pd.DataFrame([{
    "model": "XGBoost with campaign type",
    "roc_auc": roc_auc_score(y_validation, xgb_validation_probabilities),
    "average_precision": average_precision_score(
        y_validation, xgb_validation_probabilities
    ),
    "brier_score": brier_score_loss(
        y_validation, xgb_validation_probabilities
    )
}])

logistic_results = pd.read_csv(
    project_root / "reports" / "model_evaluation"
    / "logistic_regression_validation.csv"
)

model_comparison = pd.concat(
    [logistic_results, xgb_validation_metrics],
    ignore_index=True
)

model_comparison.round(4)

,model,roc_auc,average_precision,brier_score
0,Numerical baseline,0.6662,0.2145,0.1186
1,With campaign type,0.7409,0.2832,0.1138
2,XGBoost with campaign type,0.7059,0.2601,0.1149


In [8]:
import joblib

comparison_path = (
    project_root / "reports" / "model_evaluation"
    / "model_comparison_validation.csv"
)

xgb_model_path = (
    project_root / "models" / "xgboost_with_campaign_type.joblib"
)

model_comparison.to_csv(comparison_path, index=False)
joblib.dump(xgb_pipeline, xgb_model_path)

print(f"Saved comparison: {comparison_path}")
print(f"Saved XGBoost pipeline: {xgb_model_path}")

Saved comparison: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/reports/model_evaluation/model_comparison_validation.csv
Saved XGBoost pipeline: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/models/xgboost_with_campaign_type.joblib
